# JEDI SVM Baseline

**Goal:** train a Support Vector Machine classifier for `applied=yes/no`.

This notebook uses the same production-realistic feature policy as the logistic regression and random forest notebooks: no personal score columns are used as input features.

## What SVM Is

A Support Vector Machine tries to find a boundary that separates classes with the widest possible margin.

For JEDI:

```text
job features -> decision boundary -> apply / do not apply
```

We use `LinearSVC` because our feature space contains many sparse text and one-hot features. Linear SVMs often work well for text classification.

Important detail:

`LinearSVC` does not naturally output probabilities. We wrap it in `CalibratedClassifierCV` so JEDI can still produce `applied_probability` for ranking jobs.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, confusion_matrix, f1_score,
    log_loss, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
DATA_PATH = Path("data/jedi_master_prepared.csv")


## 1. Load Dataset And Target

The target is binary:

```text
yes -> 1
no  -> 0
```

In [ ]:
df = pd.read_csv(DATA_PATH)
df["applied_clean"] = df["applied"].astype(str).str.strip().str.lower()
df["target_applied"] = df["applied_clean"].map({"yes": 1, "no": 0})

if df["target_applied"].isna().any():
    raise ValueError(df.loc[df["target_applied"].isna(), "applied"].value_counts(dropna=False))

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}")
display(df[["applied", "target_applied"]].value_counts().rename("count").reset_index())
display(df.head(3))

## 2. Feature Policy

We exclude personal score columns in this notebook.

The SVM should learn from raw/standardized job facts, not from labels that you manually created during annotation.

In [ ]:
score_columns = [c for c in df.columns if c.endswith("_score")]
drop_columns = sorted(set([
    "row_id", "apply_url", "source_platform", "source_type", "scraped_at",
    "applied", "applied_clean", "target_applied", "overall_preference_score",
    "preference_reason", "near_duplicate_key", "company_name_standard", "role_title_standard",
] + score_columns))

text_source_columns = ["company_name", "role_title", "role_family", "seniority", "location", "skills_or_requirements", "skills_array"]
categorical_features = ["can_apply_now", "role_family_standard", "seniority_standard", "employment_type_standard", "work_mode_standard", "city_standard", "country_standard", "location_bucket", "experience_min_bucket", "skills_imputation_type", "freshness"]
numeric_features = ["experience_min_years", "salary_min", "salary_max", "salary_midpoint"]
boolean_features = ["has_skills_or_requirements", "is_bengaluru", "is_gurgaon_ncr", "is_india", "salary_available"]

display(pd.DataFrame({"feature": text_source_columns + categorical_features + numeric_features + boolean_features}))
display(pd.DataFrame({"dropped_column": [c for c in drop_columns if c in df.columns]}))

## 3. Build Modeling Table

SVMs work very well with TF-IDF text features. We combine job title, role, location, and skills into `job_text`.

In [ ]:
model_df = df.copy()
for col in text_source_columns:
    model_df[col] = model_df[col].fillna("").astype(str)

model_df["job_text"] = (
    model_df["company_name"] + " " + model_df["role_title"] + " " + model_df["role_family"] + " "
    + model_df["seniority"] + " " + model_df["location"] + " "
    + model_df["skills_or_requirements"] + " " + model_df["skills_array"]
)

def bool_to_int(value):
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}: return 1
    if text in {"false", "0", "no", "n"}: return 0
    return np.nan

for col in boolean_features:
    model_df[col] = model_df[col].map(bool_to_int)
for col in numeric_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

feature_columns = ["job_text"] + categorical_features + numeric_features + boolean_features
X = model_df[feature_columns].copy()
y = model_df["target_applied"].astype(int)
display(X.head(3))

## 4. Train/Test Split

We use stratification so the yes/no ratio stays stable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

def class_distribution(labels, name):
    out = labels.value_counts().rename_axis("class").reset_index(name="count")
    out["share"] = out["count"] / len(labels)
    out["set"] = name
    return out[["set", "class", "count", "share"]]

display(pd.concat([class_distribution(y_train, "train"), class_distribution(y_test, "test")], ignore_index=True))

## 5. Preprocessing Pipeline

SVMs are distance/margin-based, so numeric scaling matters more than it does for random forest.

Pipeline:

- numeric/boolean -> impute + scale
- categorical -> impute + one-hot
- text -> TF-IDF

In [ ]:
numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="constant", fill_value="missing")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3))])
text_vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=1000)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features + boolean_features),
        ("categorical", categorical_pipeline, categorical_features),
        ("text", text_vectorizer, "job_text"),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)
preprocessor

## 6. Train Calibrated Linear SVM

`LinearSVC` gives a decision score, but not probability.

`CalibratedClassifierCV` converts that decision score into a calibrated probability using cross-validation.

Important parameters:

- `C`: inverse regularization strength
- `class_weight="balanced"`: handles class imbalance
- `max_iter`: optimization iterations
- `method="sigmoid"`: probability calibration method
- `cv=3`: calibration folds

In [ ]:
base_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000,
    random_state=RANDOM_STATE,
)

calibrated_svm = CalibratedClassifierCV(
    estimator=base_svm,
    method="sigmoid",
    cv=3,
    n_jobs=-1,
)

model = Pipeline(steps=[("preprocess", preprocessor), ("classifier", calibrated_svm)])
model.fit(X_train, y_train)
print("Calibrated Linear SVM trained.")
model

## 7. Evaluate

Because we calibrated the SVM, we can use `predict_proba` for probability and ranking metrics.

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_yes": precision_score(y_test, y_pred, zero_division=0),
    "recall_yes": recall_score(y_test, y_pred, zero_division=0),
    "f1_yes": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "average_precision": average_precision_score(y_test, y_proba),
    "log_loss": log_loss(y_test, y_proba),
    "brier_score": brier_score_loss(y_test, y_proba),
}
display(pd.DataFrame([metrics]).T.rename(columns={0: "score"}))
print(classification_report(y_test, y_pred, target_names=["no", "yes"], zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, y_pred), index=["actual_no", "actual_yes"], columns=["pred_no", "pred_yes"]))

## 8. Threshold Table

SVM probabilities are calibrated estimates. We still inspect thresholds because the business tradeoff matters.

In [ ]:
threshold_rows = []
for threshold in np.arange(0.20, 0.81, 0.05):
    pred = (y_proba >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "predicted_yes_count": int(pred.sum()),
        "precision_yes": precision_score(y_test, pred, zero_division=0),
        "recall_yes": recall_score(y_test, pred, zero_division=0),
        "f1_yes": f1_score(y_test, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    })
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)
threshold_df.plot(x="threshold", y=["precision_yes", "recall_yes", "f1_yes"], marker="o", figsize=(8, 5), grid=True)
plt.title("SVM Threshold Tradeoff")
plt.show()

## 9. Cross-Validation

Calibrated SVM is heavier than logistic regression, so cross-validation may take longer.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy", "precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc", "average_precision": "average_precision"}
cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
cv_summary = []
for key, values in cv_results.items():
    if key.startswith("test_"):
        cv_summary.append({"metric": key.replace("test_", ""), "mean": np.mean(values), "std": np.std(values), "fold_scores": np.round(values, 3)})
display(pd.DataFrame(cv_summary).sort_values("metric"))

## 10. Parameter Options

Useful SVM parameters:

| Parameter | Meaning |
|---|---|
| `C` | inverse regularization strength |
| `class_weight` | class imbalance handling |
| `loss` | hinge or squared hinge objective |
| `max_iter` | optimization iterations |
| calibration `method` | sigmoid, isotonic, or temperature in newer sklearn |

Experiments to try:

```python
C=[0.1, 0.5, 1.0, 2.0, 10.0]
class_weight=[None, "balanced"]
TfidfVectorizer(max_features=[500, 1000, 2000])
```

## 11. How To Compare Against Logistic Regression And Random Forest

Compare models using the same metrics:

- ROC-AUC
- average precision
- yes recall
- yes precision
- Brier score / log loss for probability quality

SVM often performs well on text-heavy datasets. Random forest may capture interactions better. Logistic regression is usually the easiest to interpret.

In [ ]:
review_cols = ["company_name", "role_title", "role_family_standard", "seniority_standard", "location_bucket", "work_mode_standard", "employment_type_standard", "skills_or_requirements"]
test_review = df.loc[X_test.index, review_cols + ["applied"]].copy()
test_review["actual_applied_binary"] = y_test.values
test_review["predicted_applied_binary"] = y_pred
test_review["applied_probability"] = y_proba
test_review["error_type"] = np.select(
    [(test_review["actual_applied_binary"] == 1) & (test_review["predicted_applied_binary"] == 0), (test_review["actual_applied_binary"] == 0) & (test_review["predicted_applied_binary"] == 1)],
    ["false_negative", "false_positive"],
    default="correct",
)
OUTPUT_PATH = Path("data/jedi_svm_test_predictions.csv")
test_review.sort_values("applied_probability", ascending=False).to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
display(test_review.sort_values("applied_probability", ascending=False).head(15))